# Dentex - YOLOv11s Training (Google Colab)

Trains a YOLOv11-small object detector on the Dentex dental X-ray dataset
(4 classes: `Cavities`, `Damage`, `Infection`, `Wisdom`).



In [ ]:
# Install dependencies
!pip install -q ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.4 MB/s eta 0:00:00


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Unzip the dataset from Drive onto local Colab disk
import os, zipfile

DRIVE_ZIP_PATH = "/content/drive/MyDrive/Dentex.zip"  # <-- update to match your upload
DATASET_DIR = "/content/dentex"



In [ ]:
os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
    zf.extractall(DATASET_DIR)

#  Check ZIP files is there

In [ ]:
# If the zip contains a single top-level folder, descend into it so DATASET_DIR
# directly holds data.yaml, train/, valid/, test/
entries = [e for e in os.listdir(DATASET_DIR) if not e.startswith('__MACOSX')]
if len(entries) == 1 and os.path.isdir(os.path.join(DATASET_DIR, entries[0])):
    DATASET_DIR = os.path.join(DATASET_DIR, entries[0])



In [ ]:
print("Dataset root:", DATASET_DIR)
print(os.listdir(DATASET_DIR))

Dataset root: /content/dentex/Dentex
['README.roboflow.txt', 'valid', '.ipynb_checkpoints', 'data.yaml', 'test', 'README.dataset.txt', 'CLAUDE.md', 'train', 'dentex_training.ipynb']


In [ ]:
#  Rewrite data.yaml with absolute paths avoids relative-path issues in Colab VM
import yaml

data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")
with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

data_cfg["train"] = os.path.join(DATASET_DIR, "train", "images")
data_cfg["val"] = os.path.join(DATASET_DIR, "valid", "images")
data_cfg["test"] = os.path.join(DATASET_DIR, "test", "images")

colab_yaml_path = os.path.join(DATASET_DIR, "data_colab.yaml")
with open(colab_yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f)

print(data_cfg)

{'train': '/content/dentex/Dentex/train/images', 'val': '/content/dentex/Dentex/valid/images', 'test': '/content/dentex/Dentex/test/images', 'nc': 4, 'names': ['Cavities', 'Damage', 'Infection', 'Wisdom'], 'roboflow': {'workspace': 'dental-disease-detection-hpn1d', 'project': 'demo-wjml3', 'version': 8, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/dental-disease-detection-hpn1d/demo-wjml3/dataset/8'}}


In [ ]:
#  Sanity-check dataset counts
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_DIR, split, "images")
    lbl_dir = os.path.join(DATASET_DIR, split, "labels")
    n_img = len(os.listdir(img_dir))
    n_lbl = len(os.listdir(lbl_dir))
    print(f"{split}: {n_img} images, {n_lbl} labels")

train: 8167 images, 8167 labels
valid: 1290 images, 1290 labels
test: 543 images, 543 labels


In [ ]:
import os
from pathlib import Path
from collections import Counter
import yaml


In [ ]:
yaml_path = colab_yaml_path
with open(yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

In [ ]:
names = cfg["names"]
print("Clases:", names)


Clases: ['Cavities', 'Damage', 'Infection', 'Wisdom']


In [ ]:
!ls /content/dentex/Dentex/

CLAUDE.md	 data.yaml		README.dataset.txt   test   valid
data_colab.yaml  dentex_training.ipynb	README.roboflow.txt  train


In [ ]:
splits = {
    "train": data_cfg["train"],
    "val":   data_cfg["val"],
    "test":  data_cfg["test"],
}

In [ ]:
for split, img_dir in splits.items():
    # Correctly derive label_dir from img_dir which now holds absolute paths
    label_dir = Path(img_dir).parent / "labels"
    if not label_dir.exists():
        print(f"\n[{split}] labels dir no encontrado: {label_dir}")
        continue

    counter = Counter()
    label_files = list(label_dir.glob("*.txt"))

    for lf in label_files:
        with open(lf) as f:
            for line in f:
                line = line.strip()
                if line:
                    cls_id = int(line.split()[0])
                    counter[cls_id] += 1

    total = sum(counter.values())

    print(f"\n[{split}]  {len(label_files)} imagenes  |  {total} instancias totales")
    print(f"{'clase':<20} {'count':>8} {'%':>8}")
    print("-" * 38)

    for cls_id, name in enumerate(names):
        count = counter.get(cls_id, 0)
        pct = count / total * 100 if total > 0 else 0
        print(f"{name:<20} {count:>8} {pct:>7.1f}%")


[train]  8167 imagenes  |  19646 instancias totales
clase                   count        %
--------------------------------------
Cavities                 5133    26.1%
Damage                   7926    40.3%
Infection                3055    15.6%
Wisdom                   3532    18.0%

[val]  1290 imagenes  |  3435 instancias totales
clase                   count        %
--------------------------------------
Cavities                 1076    31.3%
Damage                   1873    54.5%
Infection                 451    13.1%
Wisdom                     35     1.0%

[test]  543 imagenes  |  1673 instancias totales
clase                   count        %
--------------------------------------
Cavities                  580    34.7%
Damage                    896    53.6%
Infection                 181    10.8%
Wisdom                     16     1.0%


In [ ]:
import os
import shutil
import random
from pathlib import Path
from collections import defaultdict


## Shuffle
Dataset is not balance, so I ought to  re split the instances iamges

In [ ]:

random.seed(42)

base = Path("/content/dentex/Dentex")
all_images = list((base / "train/images").glob("*.jpg")) + \
             list((base / "valid/images").glob("*.jpg")) + \
             list((base / "test/images").glob("*.jpg"))



In [ ]:
#  group images by foremost classes
class_buckets = defaultdict(list)

for img_path in all_images:
    label_path = Path(str(img_path).replace("images", "labels").replace(".jpg", ".txt"))
    if not label_path.exists():
        continue
    counter = defaultdict(int)
    with open(label_path) as f:
        for line in f:
            if line.strip():
                counter[int(line.split()[0])] += 1
    dominant = max(counter, key=counter.get) if counter else -1
    class_buckets[dominant].append(img_path)


In [ ]:

# Stratified Split 80/10/10  , -  Carefull with dominant classes
new_train, new_val, new_test = [], [], []

for cls_id, imgs in class_buckets.items():
    random.shuffle(imgs)
    n = len(imgs)
    n_val  = max(1, int(n * 0.10))
    n_test = max(1, int(n * 0.10))
    new_val.extend(imgs[:n_val])
    new_test.extend(imgs[n_val:n_val + n_test])
    new_train.extend(imgs[n_val + n_test:])

In [ ]:


print(f"train: {len(new_train)}  val: {len(new_val)}  test: {len(new_test)}")

# Copy to new folder
for split_name, split_imgs in [("train2", new_train), ("val2", new_val), ("test2", new_test)]:
    for img_path in split_imgs:
        for subdir in ["images", "labels"]:
            src = Path(str(img_path).replace("images", subdir))
            ext = ".jpg" if subdir == "images" else ".txt"
            src = src.with_suffix(ext)
            dst = base / split_name / subdir / src.name
            dst.parent.mkdir(parents=True, exist_ok=True)
            if src.exists():
                shutil.copy2(src, dst)

print("Re-split completo.")

train: 8006  val: 997  test: 997
Re-split completo.


In [ ]:
total_original = 8167 + 1290 + 543
total_nuevo = 8006 + 997 + 997
print(f"Original: {total_original}")
print(f"Nuevo:    {total_nuevo}")
print(f"Diferencia: {total_original - total_nuevo}")

Original: 10000
Nuevo:    10000
Diferencia: 0


In [ ]:
import yaml

yaml_path = "/content/dentex/Dentex/data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump({
        "train": "/content/dentex/Dentex/train2/images",
        "val":   "/content/dentex/Dentex/val2/images",
        "test":  "/content/dentex/Dentex/test2/images",
        "nc": 4,
        "names": ["Cavities", "Damage", "Infection", "Wisdom"]
    }, f)

print("data.yaml actualizado")

data.yaml actualizado


In [ ]:
# Train YOLOv11s
from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:

model = YOLO("/content/drive/MyDrive/dentex_runs/yolo11s_dentex/weights/best.pt")

In [ ]:
# Start training from  yolov11_small dentex model from tranign v1.

In [ ]:
results = model.train(
    data="/content/dentex/Dentex/data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    patience=30,
    optimizer="AdamW",
    lr0=9e-5,
    lrf=0.01,
    weight_decay=1e-4,
    warmup_epochs=3,
    warmup_bias_lr=0.0,
    cos_lr=True,
    mosaic=0.0,
    mixup=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.3,
    degrees=5.0,
    translate=0.1,
    scale=0.4,
    fliplr=0.5,
    flipud=0.0,
    device=0,
    workers=4,
    project="/content/drive/MyDrive/dentex_runs",
    name="dentex_v2",
    exist_ok=True,
    plots=True,
)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dentex/Dentex/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=9e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/dentex_runs/yolo11s_dentex/weights/best.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=dentex_v2, nbs=64, nms=False

In [ ]:
# Gotten best result at peoch 31 within
# Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.8it/s 8.9s
#                   all        997       2524      0.621      0.621       0.62      0.279#

In [ ]:
model = YOLO("/content/drive/MyDrive/dentex_runs/dentex_v2/weights/best.pt")

In [ ]:
metrics = model.val(
    data="/content/dentex/Dentex/data.yaml",
    split="val",
    imgsz=640,
    batch=32,
    device=0,
    verbose=True
)


Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,414,348 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1257.8±341.4 MB/s, size: 44.3 KB)
val: Scanning /content/dentex/Dentex/val2/labels.cache... 997 images, 17 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 997/997 181.8Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 507, len(boxes) = 2524. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 1.8it/s 17.8s
                   all        997       2524      0.632      0.613      0.616      0.282
              Cavities        318        685      0.578      0.507      0.528       0.22
                Damage        3

In [ ]:

for i, name in enumerate(["Cavities", "Damage", "Infection", "Wisdom"]):
    print(f"{name:<15} P={metrics.box.p[i]:.3f}  R={metrics.box.r[i]:.3f}  mAP50={metrics.box.ap50[i]:.3f}")


Cavities        P=0.578  R=0.507  mAP50=0.528
Damage          P=0.552  R=0.594  mAP50=0.573
Infection       P=0.557  R=0.434  mAP50=0.423
Wisdom          P=0.843  R=0.916  mAP50=0.942


In [ ]:
#  Validate the best checkpoint on the val split
best_weights = os.path.join(results.save_dir, "weights", "best.pt")
best_model = YOLO(best_weights)

metrics = best_model.val(data=colab_yaml_path, split="val")
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("per-class mAP50-95:", dict(zip(data_cfg["names"], metrics.box.maps)))

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,414,348 parameters, 0 gradients, 21.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 12.9±7.1 MB/s, size: 34.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/dentex/Dentex/valid/labels... 1290 images, 33 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1290/1290 641.1it/s 2.0s
val: New cache created: /content/dentex/Dentex/valid/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 754, len(boxes) = 3435. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 81/81 3.9it/s 

In [ ]:
# 8. Evaluate on the held-out test split
test_metrics = best_model.val(data=colab_yaml_path, split="test")
print("Test mAP50-95:", test_metrics.box.map)
print("Test mAP50:", test_metrics.box.map50)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 23.5±4.2 MB/s, size: 33.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/dentex/Dentex/test/labels... 543 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 543/543 676.9it/s 0.8s
val: New cache created: /content/dentex/Dentex/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 369, len(boxes) = 1673. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 3.5it/s 9.7s
                   all        543       1673      0.517      0.538      0.503      0.

In [ ]:
#  Export best weights (ONNX) and confirm everything landed on Drive
best_model.export(format="onnx", opset=17 )

print("Checkpoints + plots saved under:", results.save_dir)
print("best.pt:", best_weights)

Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/drive/MyDrive/dentex_runs/dentex_v2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (18.3 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 265ms
Prepared 4 packages in 1.80s
Installed 4 packages in 250ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 2.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 17...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 5.0s, saved as '/co